DEEP LEARNING 

In [1]:
!pip install torch torchvision torchaudio 

In [2]:
import pandas as pd
import numpy as np
df=pd.read_csv("powerplant_data.csv")

In [3]:
df.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [4]:
df.isnull().sum()

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

In [5]:
x=df.drop("PE",axis=1)
y=df["PE"]

In [6]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [7]:
x_train

,AT,V,AP,RH
5487,25.24,63.47,1011.30,66.21
3522,26.09,70.40,1007.41,85.37
6916,26.63,73.68,1015.15,85.13
7544,32.06,71.85,1007.90,56.44
7600,28.70,71.64,1007.11,69.85
...,...,...,...,...
5734,26.25,61.02,1011.47,71.22
5191,29.17,64.79,1016.43,61.05
5390,18.00,43.70,1015.40,61.28
860,26.73,68.84,1010.75,66.83


In [8]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

In [9]:
x_test_scaled

array([[ 1.34499288,  0.23869298, -1.28658067, -1.10532538],
       [ 0.81095912,  1.36269098, -0.74140656,  0.26485915],
       [-0.2437241 , -0.73900436,  1.99970178, -0.19713193],
       ...,
       [-0.67068342, -1.15902881, -0.29951077, -0.10651852],
       [ 1.31420898,  1.33752097, -0.87346737, -0.44288647],
       [-0.2611237 , -0.27021304,  0.37433797,  1.10646548]])

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim



In [11]:
nn

<module 'torch.nn' from 'C:\\lucky\\Lib\\site-packages\\torch\\nn\\__init__.py'>

In [12]:
x_train_tensor=torch.tensor(x_train_scaled,dtype=torch.float32)

In [13]:
y_train_tensor=torch.tensor(y_train.values,dtype=torch.float32).view(-1,1)

In [14]:
x_test_tensor=torch.tensor(x_test_scaled,dtype=torch.float32)
y_test_tensor=torch.tensor(y_test.values,dtype=torch.float32).view(-1,1)

In [15]:
from torch.utils.data import TensorDataset,DataLoader
train_dataset=TensorDataset(x_train_tensor,y_train_tensor)
test_dataset=TensorDataset(x_test_tensor,y_test_tensor)


In [16]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32)

In [17]:
import torch
import torch.nn as nn
import torch.optim as optim

# Define the class
class ANN(nn.Module):
    def __init__(self, input_size):
        super(ANN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 6),
            nn.ReLU(),
            nn.Linear(6, 6),
            nn.ReLU(),
            nn.Linear(6, 1)
        )

    def forward(self, x):
        return self.model(x)

# Make sure x_train_tensor exists before this line
input_size = x_train_tensor.shape[1]

# Instantiate the model
model = ANN(input_size)

# Loss and optimizer
crietrion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [18]:
# Train the ANN
train_losses = []
val_losses = []

best_val_loss = float("inf")

epochs = 100

for epoch in range(epochs):
    model.train()
    running_loss = 0.0 # tot training loss for 1 epoch
    
    for xb, yb in train_loader:
        # xb = features of 1 batch
        # yb = labels of 1 batch
        optimizer.zero_grad()
        
        outputs = model(xb) # forward prop....predicted outputs for this batch
        loss = crietrion(outputs, yb) # compute loss
        loss.backward() # back prop.. compute gradients
        optimizer.step() # params update
        
        running_loss += loss.item() # loss is a tensor => py float

    epoch_train_loss = running_loss / len(train_loader)
    train_losses.append(epoch_train_loss)


    # Validation
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad(): # no gradients compute
        for xb, yb in test_loader:
            outputs = model(xb)
            loss = crietrion(outputs, yb)
            running_val_loss += loss

    epoch_val_loss = running_val_loss / len(test_loader)
    val_losses.append(epoch_val_loss)

    print(f"epoch {epoch+1}/{epochs} ==> train loss = {epoch_train_loss} & val loss = {epoch_val_loss}")

epoch 1/100 ==> train loss = 204294.494921875 & val loss = 199380.703125
epoch 2/100 ==> train loss = 184030.633984375 & val loss = 161051.0625
epoch 3/100 ==> train loss = 127512.27630208334 & val loss = 92135.53125
epoch 4/100 ==> train loss = 63605.02900390625 & val loss = 41991.4921875
epoch 5/100 ==> train loss = 31210.00702718099 & val loss = 24047.630859375
epoch 6/100 ==> train loss = 20350.040934244793 & val loss = 17317.455078125
epoch 7/100 ==> train loss = 15199.019165039062 & val loss = 13050.2783203125
epoch 8/100 ==> train loss = 11297.896325683594 & val loss = 9430.4931640625
epoch 9/100 ==> train loss = 8059.570197550455 & val loss = 6600.84619140625
epoch 10/100 ==> train loss = 5545.743741861979 & val loss = 4418.4482421875
epoch 11/100 ==> train loss = 3592.9787389119465 & val loss = 2809.238037109375
epoch 12/100 ==> train loss = 2240.3229888916017 & val loss = 1741.3851318359375
epoch 13/100 ==> train loss = 1372.7863950093588 & val loss = 1074.4859619140625
epoch

In [19]:
if epoch_val_loss<best_val_loss:
    best_val_loss=epoch_val_loss
    torch.save(model.state_dict(),"good.pt")
    model.load_state_dict(torch.load("good.pt"))
    

In [20]:
model.eval()
with torch.no_grad():
    train_preds=model(x_train_tensor)
    test_preds=model(x_test_tensor)
    train_mse_loss=crietrion(train_preds,y_train_tensor)
    test_mse_loss=crietrion(test_preds,y_test_tensor)
    print("training mse:",train_mse_loss.item())
    print("testing mse:",test_mse_loss.item())
from sklearn.metrics import r2_score
print("r2 score:",r2_score(y_test,test_preds))

training mse: 20.194854736328125
testing mse: 18.54397201538086
r2 score: 0.9351935961436948


In [21]:
predicted_df=pd.DataFrame(test_preds.numpy(),columns=["predicted values"])
actual_df=pd.DataFrame(y_test.values,columns=["Actual values"])
pd.concat([predicted_df,actual_df],axis=1)

,predicted values,Actual values
0,434.962952,433.27
1,436.472321,438.16
2,461.357849,458.42
3,476.821991,480.82
4,434.433380,441.41
...,...,...
1909,451.647858,456.70
1910,431.100281,438.04
1911,468.283539,467.80
1912,430.415833,437.14


In [22]:
import pandas as pd
import numpy as np
df=pd.read_csv("DateFruit_Dataset.csv")


In [23]:
df

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.9080,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.1440,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.3940,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.2100,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.3540,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
893,255403,1925.3650,691.8453,477.1796,0.7241,570.2536,0.9785,261028,0.7269,1.4499,...,2.2423,2.3704,2.7202,-25296416768,-19168882688,-18473392128,49.0869,43.0422,42.4153,SOGAY
894,365924,2664.8230,855.4633,551.5447,0.7644,682.5752,0.9466,386566,0.6695,1.5510,...,3.4109,3.5805,3.9910,-31605219328,-21945366528,-19277905920,46.8086,39.1046,36.5502,SOGAY
895,254330,1926.7360,747.4943,435.6219,0.8126,569.0545,0.9925,256255,0.7240,1.7159,...,2.2759,2.5090,2.6951,-22242772992,-19594921984,-17592152064,44.1325,40.7986,40.9769,SOGAY
896,238955,1906.2679,716.6485,441.8297,0.7873,551.5859,0.9604,248795,0.6954,1.6220,...,2.6769,2.6874,2.7991,-26048595968,-21299822592,-19809978368,51.2267,45.7162,45.6260,SOGAY


In [24]:
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [25]:
df.shape

(898, 35)

In [26]:
x=df.drop("Class",axis=1)
y=df["Class"]
df["Class"].unique()

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [27]:
from sklearn.preprocessing import StandardScaler,LabelEncoder
le=LabelEncoder()
y=le.fit_transform(y)


In [28]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [29]:
scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

In [30]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,TensorDataset
import torch.optim as optimv

In [31]:
x_train_tensor=torch.tensor(x_train_scaled,dtype=torch.float32)
y_train_tensor=torch.tensor(y_train,dtype=torch.long)
x_test_tensor=torch.tensor(x_test_scaled,dtype=torch.float32)
y_test_tensor=torch.tensor(y_test,dtype=torch.long)

In [32]:
train_dataset=TensorDataset(x_train_tensor,y_train_tensor)
test_dataset=TensorDataset(x_test_tensor,y_test_tensor)

In [33]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32)

In [34]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()
        self.model=nn.Sequential(
        nn.Linear(x.shape[1],64),
        nn.ReLU(),
        nn.Linear(64,64),
        nn.ReLU(),
        nn.Linear(64,7),
        )
    def forward(self,x):
        return self.model(x)
model=ANN()
criteria=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())


In [35]:
epochs=100
for epoch in range(epochs):
    model.train()
    running_loss=0.0
    for xb,yb in train_loader:
        optimizer.zero_grad()
        outputs=model(xb)
        loss=criteria(outputs,yb)
        loss.backward()
        optimizer.step()
        running_loss+=loss.item()
    train_loss=running_loss/len(train_loader)
    print(f"epoch={epoch+1}/{epochs};loss={train_loss}")


epoch=1/100;loss=1.6631891364636628
epoch=2/100;loss=1.120390930901403
epoch=3/100;loss=0.7432067018488179
epoch=4/100;loss=0.5346775819425997
epoch=5/100;loss=0.433590283860331
epoch=6/100;loss=0.3807770428450211
epoch=7/100;loss=0.3371939153774925
epoch=8/100;loss=0.30387758625590283
epoch=9/100;loss=0.2826885360738505
epoch=10/100;loss=0.2675750320372374
epoch=11/100;loss=0.23810335216314896
epoch=12/100;loss=0.23149245945007904
epoch=13/100;loss=0.2217030810273212
epoch=14/100;loss=0.20273739585409994
epoch=15/100;loss=0.19371194100898245
epoch=16/100;loss=0.1792951222995053
epoch=17/100;loss=0.18355905135040698
epoch=18/100;loss=0.16852495495391928
epoch=19/100;loss=0.1674412634709607
epoch=20/100;loss=0.15260840320716734
epoch=21/100;loss=0.15744928302972214
epoch=22/100;loss=0.1387444572604221
epoch=23/100;loss=0.14601245068985483
epoch=24/100;loss=0.1344725653205229
epoch=25/100;loss=0.1299099242719619
epoch=26/100;loss=0.12732319485234178
epoch=27/100;loss=0.12160007185910059


In [36]:
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb)
        _, predicted = torch.max(outputs, 1)

        correct += (predicted == yb).sum().item()
        total += yb.size(0)

print("total vals:", total)
print("correct values:", correct)
print("accuracy:", correct / total * 100)

total vals: 180
correct values: 171
accuracy: 95.0


In [37]:

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10

In [38]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms


In [39]:
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

In [40]:
trainset=CIFAR10(root="AIML.ipynb",train=True,download=True,transform=transform)

In [41]:
testset=CIFAR10(root="AIML.ipynb",train=False,download=True,transform=transform)

In [42]:
trainloader=DataLoader(trainset,batch_size=64,shuffle=True)

In [43]:
test_loader=DataLoader(testset,batch_size=64)

In [44]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        self.conv_layers=nn.Sequential(
        nn.Conv2d(3,32,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),
        nn.Conv2d(32,64,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),
        nn.Conv2d(64,128,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),)
        self.fc_layers=nn.Sequential(nn.Linear(4*4*128, 256),
        nn.ReLU(),
        nn.Linear(256,10)
        )
    def forward(self,x):
        x=self.conv_layers(x)
        x=x.view(x.size(0),-1)
        x=self.fc_layers(x)
        return x
     

In [45]:
model=CNN()
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [46]:
epochs = 1

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()
        
        output = model.forward(images) # FP
        loss = criterion(output, labels) # loss fnx
        loss.backward() # BP
        optimizer.step() # update params

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

epoch=1/1 & loss=1.3679680120762048


In [47]:
correct_labels=0
total_labels=0
model.eval()
with torch.no_grad():
    for images,labels in test_loader:
        outputs=model.forward(images)
        _,predicted=torch.max(outputs,1)
        correct_labels+=(predicted==labels).sum().item()
        total_labels+=labels.size(0)
    print(f"accuracy={correct_labels/total_labels*100}")

accuracy=62.11


In [48]:
import pandas as pd
df=pd.read_csv("IMDB dataset.csv")

In [49]:
df.shape


(50000, 2)

In [50]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [51]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [52]:
df.drop_duplicates(inplace=True)

In [53]:
df.shape

(49582, 2)

In [54]:
df["review"]=df["review"].str.lower()

In [55]:
import re


In [56]:
def remove_urls(text):
    text=re.sub(r"http\S+","",text)
    return text
df["review"]=df["review"].apply(remove_urls)


In [57]:
def remove_puntuations(text):
    text=re.sub(r"[^A-Za-z0-9/s]","",text)
    return text
df["review"]=df["review"].apply(remove_puntuations)


In [58]:
def remove_html(text):
    text=re.sub(r"<.*?>","",text)
    return text
df["review"]=df["review"].apply(remove_html)


In [59]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')  # new requirement in recent NLTK versions
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sohai\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\sohai\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sohai\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [60]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [61]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")

    return text

df["review"] = df["review"].apply(remove_stopwords)

In [62]:
df.head()

,review,sentiment
0,oneoftheotherreviewershasmentionedthatafterwat...,positive
1,awonderfullittleproductionbr/br/thefilmingtech...,positive
2,ithoughtthiswasawonderfulwaytospendtimeonatooh...,positive
3,basicallytheresafamilywherealittleboyjakethink...,negative
4,pettermatteisloveinthetimeofmoneyisavisuallyst...,positive


In [63]:
from nltk.stem import PorterStemmer
def stemming(text):
    ps=PorterStemmer()
    stemmed_words=[]
    tokens=word_tokenize(text)
    for token in tokens:
        stemmed_token=ps.stem(token)
        stemmed_words.append(stemmed_token)
        return "".join(stemmed_words)
df["review"]=df["review"].apply(stemming)


In [64]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df["sentiment"]=le.fit_transform(df["sentiment"])
y=df["sentiment"]


In [65]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

In [66]:
df.head()

,review,sentiment
0,oneoftheotherreviewershasmentionedthatafterwat...,1
1,awonderfullittleproductionbr/br/thefilmingtech...,1
2,ithoughtthiswasawonderfulwaytospendtimeonatooh...,1
3,basicallytheresafamilywherealittleboyjakethink...,0
4,pettermatteisloveinthetimeofmoneyisavisuallyst...,1


In [67]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [68]:
tf=TfidfVectorizer(max_features=5000)
x=tf.fit_transform(df["review"])

In [69]:
print(x)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 38633 stored elements and shape (49582, 5000)>
  Coords	Values
  (0, 4395)	1.0
  (1, 4395)	1.0
  (2, 4395)	1.0
  (3, 4395)	0.38318966659021236
  (3, 2115)	0.923669680902476
  (4, 4395)	1.0
  (7, 4395)	1.0
  (8, 4395)	1.0
  (9, 4395)	1.0
  (10, 4395)	1.0
  (11, 4395)	1.0
  (12, 4395)	1.0
  (13, 4395)	1.0
  (17, 4395)	0.7907250346260184
  (17, 4472)	0.6121714789302767
  (18, 2880)	1.0
  (20, 4395)	1.0
  (21, 4395)	1.0
  (23, 4395)	0.3042192666360845
  (23, 115)	0.5499850415564055
  (23, 3712)	0.5499850415564055
  (23, 171)	0.5499850415564055
  (24, 4395)	1.0
  (25, 4395)	1.0
  (26, 4395)	0.7032254898933338
  :	:
  (49548, 239)	1.0
  (49555, 4395)	1.0
  (49556, 4395)	1.0
  (49557, 4395)	1.0
  (49559, 4395)	1.0
  (49560, 3109)	1.0
  (49561, 4395)	1.0
  (49562, 4395)	1.0
  (49563, 4395)	0.47737282246510043
  (49563, 23)	0.8787008526066785
  (49564, 4395)	1.0
  (49565, 4395)	1.0
  (49567, 4395)	1.0
  (49569, 4395)	1.0
  (49570, 43

In [70]:
x

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 38633 stored elements and shape (49582, 5000)>

In [71]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [72]:
x_train.shape

(39665, 5000)

In [73]:
import torch
from torch.utils.data import TensorDataset,DataLoader


In [74]:
x_train=x_train.toarray()
x_test=x_test.toarray()


In [75]:
train_set=TensorDataset(
    torch.from_numpy(x_train).float(),
    torch.from_numpy(y_train.values).float()
)
test_set=TensorDataset(
    torch.from_numpy(x_test).float(),
    torch.from_numpy(y_test.values).float())


In [76]:
train_loader=DataLoader(train_set,shuffle=True,batch_size=64)
test_loader=DataLoader(test_set,shuffle=True,batch_size=64)

In [77]:
import torch.nn as nn
import torch.optim as optim

In [78]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layers=1):
        super().__init__()
        self.hidden_size=hidden_size
        self.num_layers=num_layers
        self.rnn=nn.RNN(input_size,hidden_size,num_layers,batch_first=True)
        self.fc=nn.Linear(hidden_size,1)
    def forward(self,x):
        h0=torch.zeros(self.num_layers,x.size(0),self.hidden_size)
        out,_=self.rnn(x,h0)
        out=self.fc(out[:,-1,:])
        return out

In [79]:
input_size=x_train.shape[1]
model=RNN(input_size)
criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters())

In [80]:
epochs=10
for epoch in range(epochs):
    model.train()
    for xb,yb in train_loader:
        optimizer.zero_grad()
        xb=xb.unsqueeze(1)
        outputs=model(xb)
        outputs=torch.sigmoid(outputs.squeeze())
        loss=criterion(outputs,yb)
        loss.backward()
        optimizer.step()
    print(f"epoch ={epoch+1}/{epochs} and loss={loss.item()}")

epoch =1/10 and loss=0.6916674971580505
epoch =2/10 and loss=0.6998897194862366
epoch =3/10 and loss=0.6167159080505371
epoch =4/10 and loss=0.6953518986701965
epoch =5/10 and loss=0.6486122012138367
epoch =6/10 and loss=0.6406015753746033
epoch =7/10 and loss=0.6292611956596375
epoch =8/10 and loss=0.6095796823501587
epoch =9/10 and loss=0.6459051370620728
epoch =10/10 and loss=0.6515705585479736


In [81]:
model.eval()
with torch.no_grad():
    correct_vals=0
    tot_vals=0
    for xb,yb in test_loader:
        xb=xb.unsqueeze(1)
        outputs=model(xb)
        predicted=(torch.sigmoid(outputs.squeeze())>0.5).float()
        tot_vals+=yb.size(0)
        correct_vals+=(predicted==yb).sum().item()
    print(f"accuracy={correct_vals/tot_vals*100}")

accuracy=51.95119491781789


In [82]:
!pip install gymnasium

In [83]:
!pip install "gymnasium[toy-text]"

In [ ]:
import gymnasium as gym
import numpy as np
import random

In [ ]:
env=gym.make("CliffWalking-v1")

In [ ]:
env

In [ ]:
print(env.observation_space.n)

In [ ]:
print(env.action_space.n)